# Chunk Happens

Time to define vectors for the IoT book -- but the *way* we do it matters more than it looks.

Part 2 gave the whole book to an agent by pasting its text straight into the instructions, on every single call. It worked, but it's expensive: the model has to read and make sense of the entire book from scratch each time, which costs a lot of tokens for a question that might only need one paragraph of it.

Part 7 suggests an obvious fix: embed the book the same way we embedded a sentence -- one vector for the whole thing, then compare it to the question's vector. Two problems kill that idea immediately:

1. **Dimension.** A sentence's meaning fits reasonably well into a few hundred numbers. A 218-page book covering dozens of protocols, architectures and design trade-offs doesn't -- squeezing all of that into a vector the same size would need an enormous embedding space just to keep it distinguishable from any other book.
2. **Precision.** Even with a huge enough space, a single vector can only ever answer "how close is this whole book to the question, overall?" It has no way to point at *which part* of the book actually answers it -- and that's the whole point here: not a vague gist of the source, but access to the specific passage that's actually relevant.

So instead of one vector for the whole book, we're going to build a *set* of them -- one per passage, small enough to be precise, many enough to cover everything. That also means hundreds of embeddings instead of one, which is exactly where Part 7's local model earns its keep over a hosted API: `multilingual-e5-small`, recreated below exactly as it was, so nothing gets re-paid on every rebuild of the index.

## Chunking, in practice

`documents/PLIDO_BOOK_en.pdf` is 218 pages, roughly 75,000 words. Beyond the conceptual reasons above, there's also a hard technical one: the embedding model has a **512-token input limit**, so anything past roughly the first 350 words of whatever we feed it is silently discarded. Not an error, not a warning -- just truncated, and you'd never know from the output.

So we **chunk**: cut the text into passages small enough to embed intact, each one specific enough to be a meaningful answer on its own. Two parameters matter:

* **Size** -- 180 words here, comfortably inside the 512-token limit even with long technical words. Too big and you hit the truncation trap; too small and a passage loses the context that makes it meaningful.
* **Overlap** -- 40 words repeated between consecutive chunks, so a sentence that happens to straddle a boundary still appears whole in one of them.

<br>
<img src="images/chunk_overlap_academic.jpg" width="550" alt="Chunking with Overlap Diagram" style="display: block; margin: 15px auto;">
<br>

One more thing this particular PDF forces on us. A book has a table of contents and an index -- pages of `LoRaWAN . . . . . . . . 27, 59`. Those chunks are almost pure punctuation, they carry no meaning, and they pollute results. So we drop any chunk that isn't mostly letters -- and rather than take that on faith, Program 1 below prints a couple of the actual chunks it drops, so you can see for yourself.

That filter isn't perfect, though: a hex dump or a garbled OCR'd code listing can easily be over 60% letters while still being noise. Program 1 also checks each chunk's embedding against the previous one's -- since consecutive chunks share 40 overlapping words, they're normally quite similar (0.91 on average here), so the *lowest* similarities are worth a look: they tend to land right where prose gives way to exactly this kind of leftover noise.

In [ ]:
# Program 1: recreate the local embedding model from Part 7, then chunk and index the IoT book

import os
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from pypdf import PdfReader
from dotenv import load_dotenv

load_dotenv(override=True)

embed_name = "intfloat/multilingual-e5-small"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_name)
embed_model = AutoModel.from_pretrained(embed_name)
embed_model.eval()

def embed(texts, batch_size=32):
    """Same mean-pooling as Part 7, plus length-1 normalisation, batched so a few hundred
    passages don't take a few hundred forward passes."""
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = embed_tokenizer(texts[start:start + batch_size], return_tensors="pt",
                                truncation=True, max_length=512, padding=True)
        with torch.no_grad():
            hidden_states = embed_model(**batch).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).float()
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        vectors.append(F.normalize(pooled, dim=-1))
    return torch.cat(vectors)

def embed_passages(texts):
    return embed([f"passage: {t}" for t in texts])

def embed_query(text):
    return embed([f"query: {text}"])[0]

DOCS_DIR = os.path.abspath(os.path.join(os.getcwd(), "documents"))

book = PdfReader(os.path.join(DOCS_DIR, "PLIDO_BOOK_en.pdf"))
book_text = "\n".join((page.extract_text() or "") for page in book.pages)
print(f"Book: {len(book.pages)} pages, {len(book_text.split()):,} words")

def chunk_text(text, size=180, overlap=40):
    """Cut text into overlapping passages of `size` words."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + size]))
        start += size - overlap
    return chunks

def is_useful(chunk):
    """Drop table-of-contents and index chunks: mostly dots and page numbers, few real words."""
    letters = sum(character.isalpha() for character in chunk)
    return letters / max(len(chunk), 1) > 0.6

raw_chunks = chunk_text(book_text)
book_chunks = [c for c in raw_chunks if is_useful(c)]
dropped_chunks = [c for c in raw_chunks if not is_useful(c)]
print(f"{len(raw_chunks)} chunks -> {len(book_chunks)} kept "
      f"({len(dropped_chunks)} dropped as table-of-contents/index)")

print("\nA couple of the chunks that got dropped, to see why:")
for chunk in dropped_chunks[:2]:
    letters = sum(character.isalpha() for character in chunk)
    ratio = letters / max(len(chunk), 1)
    print(f"  ({ratio:.0%} letters) {' '.join(chunk.split())[:150]!r}")

start_time = time.time()
book_vectors = embed_passages(book_chunks)
print(f"\nIndexed in {time.time() - start_time:.0f}s, entirely on this machine -- zero API calls.")

# Consecutive chunks share 40 overlapping words, so they're normally quite similar --
# the lowest similarities are worth a look, since they tend to mark leftover extraction
# noise that is_useful() let through (over 60% letters, but still not real prose).
consecutive_similarity = [(book_vectors[i] @ book_vectors[i - 1]).item() for i in range(1, len(book_chunks))]
print(f"\nChunk-to-chunk similarity: {sum(consecutive_similarity) / len(consecutive_similarity):.2f} average")

lowest = sorted(range(len(consecutive_similarity)), key=lambda i: consecutive_similarity[i])[:2]
print("The 2 lowest -- where is_useful() still let noise through:")
for i in lowest:
    print(f"  {consecutive_similarity[i]:.3f}  ...{book_chunks[i][-60:]!r}")
    print(f"          {book_chunks[i + 1][:60]!r}...")

## Retrieving from the index

The index is built. Retrieval is now the cosine similarity from Part 4, applied at scale: embed the question, compare it to every stored chunk, keep the closest few.

Because the vectors are normalised, that whole comparison is a single matrix multiplication -- `vectors @ question`, one dot product per chunk, done in one call rather than a Python loop over hundreds of entries. This is the same operation a real vector database (FAISS, Chroma, pgvector...) optimises for millions of chunks; at our scale, plain PyTorch is entirely enough.

Note we ask for the top **three** chunks, not just the best one. Retrieval isn't perfect, and giving the model a few candidates lets it pick out the relevant part itself -- a cheap and very effective safety margin.

In [ ]:
# Program 2: retrieve the passages closest to a question

def retrieve(question, chunks, vectors, top_k=3):
    question_vector = embed_query(question)
    scores = vectors @ question_vector          # one dot product per chunk, in one operation
    best = scores.topk(top_k)
    return [(scores[i].item(), chunks[i]) for i in best.indices.tolist()]

for question in ["How does 6LoWPAN compress IPv6 headers?",
                 "Qu'est-ce que le protocole MQTT ?"]:
    print(f"Q: {question}")
    for score, chunk in retrieve(question, book_chunks, book_vectors):
        print(f"  {score:.3f}  {' '.join(chunk.split())[:150]}...")
    print()

Both questions land on the right passage, and the second one is worth a second look: the question is in French, the book is in English, and they share almost no vocabulary -- yet the MQTT passage comes back. That's the multilingual embedding doing exactly what Part 7's Program 1.2 found `SmolLM2` couldn't: `multilingual-e5-small`, recreated in Program 1 above, was actually trained for this, now proving it across 400 passages of a real book instead of six toy sentences.

## Wrapping retrieval as a tool

Like every other capability in this course, retrieval becomes useful to an agent once it's wrapped as a tool. This one doesn't fetch a web page or write a file -- it searches the index we just built, and hands back raw passages for the model to answer from.

Compare this with Part 2's `book_agent`, which put the *entire* book into its instructions on every single call. Same book, same questions, but now the model only ever sees the three passages that matter.

In [ ]:
# Program 3: a RAG agent over the book -- it only ever sees the passages it retrieves

from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

RENNES_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/instance@imt/api/"
rennes_client = AsyncOpenAI(base_url=RENNES_BASE_URL, api_key=os.environ["RENNES_API_KEY"])
rennes_model = OpenAIChatCompletionsModel(model="ilaas/mistral-small-4-119b", openai_client=rennes_client)

@function_tool
def search_book(question: str):
    """Search the Internet of Things book for the passages most relevant to a question."""
    print(f"[search_book] the model asked: {question!r}")
    passages = retrieve(question, book_chunks, book_vectors)
    for score, chunk in passages:
        print(f"[search_book]   -> {score:.3f}  {' '.join(chunk.split())[:100]}...")
    return "\n\n---\n\n".join(chunk for _, chunk in passages)

book_rag_agent = Agent(
    name="PLIDO Book RAG Agent",
    instructions="Answer questions about the Internet of Things using the search_book tool. "
                 "Base your answer only on the passages it returns -- if they don't contain "
                 "the answer, say so rather than inventing one.",
    model=rennes_model,
    tools=[search_book],
)

result = await Runner.run(book_rag_agent, "What is 6LoWPAN and why is it needed?", max_turns=6)
display(Markdown(result.final_output))

A correct, grounded answer, built from three passages instead of 75,000 words. That's RAG working exactly as advertised -- and the `[search_book]` lines printed above show exactly how it got there: the question the model actually sent to the tool (not always word-for-word what you asked it), and the three passages that came back for it to answer from.

It worked this smoothly for a reason worth naming: the book is a *clean* corpus. One document, written by one author, no navigation menus, no duplicated pages, consistent structure throughout. Real-world corpora rarely are -- that's Part 9.

### Your turn: chat with the book

Instead of editing a `question = "..."` string and re-running a cell, wrap `book_rag_agent` in a chat interface -- same `gradio.ChatInterface` pattern as Part 2 and Part 9's Program 4. Watch the `[search_book]` trace printed below as you chat: it shows exactly what question the model sent to the nearest-vector search, and which passages came back for it.

In [ ]:
# Program 4: chat with the book -- watch the nearest-vector search happen live

import gradio

async def chat_async(message, history):
    try:
        result = await Runner.run(book_rag_agent, message, max_turns=6)
        return result.final_output
    except Exception as e:
        return f"Error: {e}"

gradio.ChatInterface(chat_async, title="PLIDO Book RAG Agent").launch()

## LangChain

Program 1 called `AutoModel`, ran mean-pooling by hand, and managed `torch` tensors directly the whole way through. In practice, most real RAG code doesn't -- **LangChain** and a vector store like **Chroma** wrap the entire pipeline behind a few high-level calls: split a document, embed it, store it, query it, without a single tensor ever appearing in your code. Let's redo Program 1 and Program 2 that way, on the exact same book.

In [ ]:
# Program 5: the same book, chunked, embedded and persisted with LangChain + Chroma

from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

class LocalEmbeddings(Embeddings):
    """Wraps Program 1's embed_passages/embed_query behind the two methods LangChain's
    Embeddings interface expects -- no new model, no new library, just an adapter."""
    def embed_documents(self, texts):
        return embed_passages(texts).tolist()
    def embed_query(self, text):
        return embed([f"query: {text}"])[0].tolist()

# RecursiveCharacterTextSplitter counts characters, not words like our chunk_text --
# ~1000 characters lands in the same ballpark as our 180-word chunks.
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
langchain_chunks = splitter.split_text(book_text)
print(f"RecursiveCharacterTextSplitter: {len(langchain_chunks)} chunks "
      f"(vs {len(book_chunks)} from our own chunk_text -- it has no is_useful step, "
      f"so table-of-contents junk is still in there)")

# persist_directory is the whole point of a vector *database*: this writes the
# embeddings to disk under ./chroma_book_index, not just into a Python variable.
start_time = time.time()
vectorstore = Chroma.from_texts(langchain_chunks, embedding=LocalEmbeddings(),
                                persist_directory="chroma_book_index")
print(f"Indexed and persisted in {time.time() - start_time:.0f}s")

for question in ["How does 6LoWPAN compress IPv6 headers?",
                 "Qu'est-ce que le protocole MQTT ?"]:
    print(f"\nQ: {question}")
    for doc in vectorstore.similarity_search(question, k=3):
        print(f"  {' '.join(doc.page_content.split())[:150]}...")

Same book, same answers in substance -- the 6LoWPAN result leads with header compression and fragmentation, the MQTT results are all genuinely about publish/subscribe. `LocalEmbeddings` is the entire integration cost: two methods, both one-liners calling code Program 1 already wrote. Everything else -- splitting, storing, comparing vectors, ranking results -- is `RecursiveCharacterTextSplitter` and `Chroma` doing what `chunk_text`, `book_vectors` and `retrieve` did by hand.

Two real differences are worth noticing, not just glossed over:

* **The chunking step alone comes with several ready-made strategies** instead of hand-written functions: `RecursiveCharacterTextSplitter` (used here, tries paragraph and sentence boundaries before falling back to a raw character count), a Markdown/HTML-header-aware splitter (the packaged version of Part 9's `chunk_structured`), and a semantic splitter that cuts wherever consecutive sentences stop being similar -- the same idea Program 1's chunk-to-chunk similarity check was doing by hand, turned into an actual chunking strategy instead of just a sanity check.
* **The convenience has a cost.** `langchain_chunks` skipped our `is_useful` filter entirely, because `RecursiveCharacterTextSplitter` has no idea a table of contents looks different from a paragraph -- it still worked here because the junk was a small fraction of a 218-page book, but on a smaller or messier document it wouldn't. Part 9's actual fix, cutting on the TAF catalogue's own section headings, needed the freedom to write a chunker for that one document's specific format; a library only gets you there if it happens to ship the right splitter already, or if you're willing to open its source and extend it. Knowing what `chunk_text`, `embed_passages` and `retrieve` actually do, which is what these two programs were for, is what makes that call possible either way.

### Chroma is a database, not just a Python object

Everything so far reused the `vectorstore` variable Program 5 left behind -- which doesn't actually prove Chroma persisted anything, since a plain Python dictionary would survive just as well within the same running notebook. The real test: open a **new** connection to `chroma_book_index`, with none of Program 5's embedding work repeated, and check it already has everything on disk.

In [ ]:
# Program 5.1: reconnect to the persisted index -- prove it doesn't re-embed anything

start_time = time.time()
reloaded_vectorstore = Chroma(persist_directory="chroma_book_index", embedding_function=LocalEmbeddings())
chunk_count = len(reloaded_vectorstore.get()["ids"])
print(f"Reconnected in {time.time() - start_time:.1f}s -- "
      f"{chunk_count} chunks already on disk (compare to Program 5's ~20s to build them)")

for doc in reloaded_vectorstore.similarity_search("How does 6LoWPAN compress IPv6 headers?", k=1):
    print(f"  {' '.join(doc.page_content.split())[:150]}...")

### Wrapping the Chroma index as a tool, too

Same move as Program 3: an LLM never touches `vectorstore` directly, it calls a tool that does. `search_book_langchain` wraps `vectorstore.similarity_search` exactly the way `search_book` wrapped `retrieve` -- and then it's the same `gradio.ChatInterface` pattern as Program 4 to actually talk to it.

In [ ]:
# Program 6: wrap the Chroma index as a tool, and chat with the LangChain version

@function_tool
def search_book_langchain(question: str):
    """Search the Internet of Things book (LangChain/Chroma index) for the passages
    most relevant to a question."""
    print(f"[search_book_langchain] the model asked: {question!r}")
    docs = vectorstore.similarity_search(question, k=3)
    for doc in docs:
        print(f"[search_book_langchain]   -> {' '.join(doc.page_content.split())[:100]}...")
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

langchain_book_agent = Agent(
    name="LangChain Book RAG Agent",
    instructions="Answer questions about the Internet of Things using the "
                 "search_book_langchain tool. Base your answer only on the passages it "
                 "returns -- if they don't contain the answer, say so rather than "
                 "inventing one.",
    model=rennes_model,
    tools=[search_book_langchain],
)

async def chat_async(message, history):
    try:
        result = await Runner.run(langchain_book_agent, message, max_turns=6)
        return result.final_output
    except Exception as e:
        return f"Error: {e}"

gradio.ChatInterface(chat_async, title="LangChain Book RAG Agent").launch()

## Key takeaways

<img src="images/takeaways.jpg" width="150" alt="Key takeaways" style="float: left; margin-right: 15px; margin-bottom: 10px;">

* **RAG** means embedding source material once, in advance, then using cosine similarity to retrieve only the passages relevant to a question -- instead of stuffing everything into every prompt, which is what Part 2's `book_agent` did.
* Retrieval is a third kind of decision-maker alongside Part 6's algorithmic and agentic ones: relevance decided by nearest-neighbour search over embeddings, not by fixed Python branches and not by an LLM reading everything.
* **One vector per document doesn't work at scale.** A single embedding can only say how close a whole document is to a question overall -- it can't point at which part answers it, which is the entire reason to chunk instead.
* **Bulk embedding belongs on a local model.** A hosted API is right for occasional calls, but indexing a corpus means hundreds of them, re-paid on every rebuild -- enough to exhaust a free tier for no benefit, which is why everything here reused Part 7's `multilingual-e5-small` instead of Gemini.
* **Every embedding model has a hard input limit** (512 tokens here). Exceed it and your text is silently truncated -- no error, no warning, just half your chunk quietly discarded.
* **LangChain doesn't need a different model.** `Chroma` and `RecursiveCharacterTextSplitter` replace `book_vectors`/`chunk_text`, but the embeddings themselves still come from Program 1's own `embed_passages`/`embed_query`, wrapped in a two-method adapter -- the library changes the plumbing, not the model doing the actual understanding.
* **Chroma is an actual database, not a variable.** `persist_directory` writes the embeddings to disk; Program 5.1 opens a brand new connection and queries it without repeating a single second of Program 5's embedding work -- exactly what `book_vectors` alone can't do once the notebook's Python process ends.
* **The tool-wrapping pattern doesn't change either.** Whether the search behind it is a hand-written `retrieve()` or `vectorstore.similarity_search()`, the LLM only ever sees a `@function_tool` and a question -- Program 3/4 and Program 6 are the same agent-and-chat shape around two different retrieval implementations.
* On a clean, well-structured document, fixed-size chunking with a bit of overlap is enough to get accurate, grounded retrieval -- Part 9 tests whether that still holds once the corpus gets messier.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `AutoTokenizer.from_pretrained`, `AutoModel.from_pretrained`, `Agent`, `Runner.run`, and `@function_tool` -- all introduced in Part 7):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `AutoModel.from_pretrained(name)` (`transformers`) | a model name | a model exposing raw hidden states, no next-token head | Program 1 |
| `vectors @ query` (`torch`) | a matrix of chunk vectors, one query vector | one similarity score per chunk, in a single operation | Program 2 |
| `tensor.topk(k)` (`torch`) | how many results to keep | the k highest scores and their indices | Program 2 |
| `PdfReader(path_or_bytes)` (`pypdf`) | a file path, or a `BytesIO` of downloaded bytes | a PDF whose `.pages` expose `.extract_text()` | Program 1 |
| `gradio.ChatInterface(fn).launch()` (`gradio`) | an async function taking `(message, history)` | a running local chat server | Programs 4, 6 |
| `Embeddings` (`langchain_core.embeddings`) | subclass with `embed_documents`/`embed_query` | an object any LangChain vector store accepts | Program 5 |
| `RecursiveCharacterTextSplitter(...).split_text(text)` (`langchain_text_splitters`) | `chunk_size`, `chunk_overlap`, a string | a list of text chunks | Program 5 |
| `Chroma.from_texts(texts, embedding=..., persist_directory=...)` (`langchain_chroma`) | chunks, an `Embeddings` instance, a folder path | a queryable vector store, written to disk | Program 5 |
| `Chroma(persist_directory=..., embedding_function=...)` (`langchain_chroma`) | a folder path, an `Embeddings` instance | a connection to an *existing* index, no re-embedding | Program 5.1 |
| `vectorstore.similarity_search(query, k=)` (`langchain_chroma`) | a query string, how many results | the k closest `Document`s (`.page_content`) | Programs 5, 5.1, 6 |